## Fine-tuned DistilRoBERTa-base for Emotion Classification 🤬🤢😀😐😭😲

DistilRoBERTa-base is a transformer model that performs sentiment analysis. I download this fine-tuned model on transcripts from the Friends show with the goal of classifying emotions from text data, specifically dialogue from Netflix shows or movies. The model predicts 6 Ekman emotions and a neutral class. These emotions include anger, disgust, fear, joy, neutrality, sadness, and surprisre.

The model is a fine-tuned version of Emotion English DistilRoBERTa-base and DistilRoBERTa-base. This model was initially trained on the following table from Emotion English DistilRoBERTa-base:

| Name                                         | anger | disgust | fear | joy | neutral | sadness | surprise |
|----------------------------------------------|:-----:|:-------:|:----:|:---:|:-------:|:--------:|:--------:|
| Crowdflower (2016)                           | Yes   |   -     |  -   | Yes |   Yes   |   Yes    |   Yes    |
| Emotion Dataset, Elvis et al. (2018)         | Yes   |   -     | Yes  | Yes |    -    |   Yes    |   Yes    |
| GoEmotions, Demszky et al. (2020)            | Yes   |  Yes    | Yes  | Yes |   Yes   |   Yes    |   Yes    |
| ISEAR, Vikash (2018)                         | Yes   |  Yes    | Yes  | Yes |    -    |   Yes    |    -     |
| MELD, Poria et al. (2019)                    | Yes   |  Yes    | Yes  | Yes |   Yes   |   Yes    |   Yes    |
| SemEval-2018, EI-reg, Mohammad et al. (2018) | Yes   |   -     | Yes  | Yes |    -    |   Yes    |    -     |

It was fine-tuned on:

| Name                      | anger | disgust | fear | joy | neutral | sadness | surprise |
|---------------------------|:-----:|:-------:|:----:|:---:|:-------:|:--------:|:--------:|
| Emotion Lines (Friends)   | Yes   |  Yes    | Yes  | Yes |   Yes   |   Yes    |   Yes    |

In [18]:
from flask import Flask, render_template, request
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [19]:
app = Flask(__name__)

In [20]:
# Load the pre-trained model and tokenizer from Hugging Face
model_name = "michellejieli/emotion_text_classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# List of emotion labels the model predicts
labels = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [21]:
# Prediction function: takes text input and returns the predicted emotion
def predict_emotion(text):
    # Tokenize input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    # Perform the prediction without computing gradients
    with torch.no_grad():
        logits = model(**inputs).logits
    # Get the index of the maximum predicted value (class)
    predicted_class_id = logits.argmax().item()
    return labels[predicted_class_id]

In [22]:

# Route for the home page
@app.route('/', methods=['GET', 'POST'])
def home():
    result = None
    if request.method == 'POST':
        # Get the text from the form
        text = request.form['text']
        # Make the emotion prediction
        result = predict_emotion(text)
    # Render the template and pass the prediction result to the page
    return render_template('index.html', result=result)

In [23]:
# Run the app if this script is executed directly
if __name__ == '__main__':
    app.run(debug=True, port=5001)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
 * Restarting with stat
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 1074, in launch_instance
    app.initialize(argv)
  File "/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 118, in inner
    return method(app, *args, **kwarg

SystemExit: 1

Michellejieli. (n.d.). Emotion text classifier [Modelo de aprendizaje automático]. Hugging Face. https://huggingface.co/michellejieli/emotion_text_classifier